# Monthly evaluation notebook

This notebook evaluates monthly outputs for two modes:
- **agent**: outputs include analyst indicators and manager actions
- **workflow**: outputs include manager recommendations only, prices are joined from the gold panel

Stage 1 follows Thiago style scaling, MinMaxScaler fitted on gold values.
Stage 2 is an optional trading simulation based on BUY, SELL, HOLD actions.


In [2]:
# Imports
import json
import math
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler


## 1. Helper functions

In [3]:
def first_trading_day_per_month(df: pd.DataFrame, date_col: str = "date") -> pd.DataFrame:
    """Keep exactly one row per ticker per calendar month, picking the first trading day."""
    out = df.copy()
    out[date_col] = pd.to_datetime(out[date_col], errors="coerce")
    out = out.dropna(subset=[date_col])

    out["month"] = out[date_col].dt.to_period("M").astype(str)
    out["ticker"] = out["ticker"].astype(str).str.upper().str.strip()

    out = out.sort_values(["ticker", date_col]).reset_index(drop=True)
    out = out.drop_duplicates(subset=["ticker", "month"], keep="first").reset_index(drop=True)
    out[date_col] = out[date_col].dt.strftime("%Y-%m-%d")
    return out


def load_gold_monthly_panel(path: Path) -> pd.DataFrame:
    """Load the gold panel and collapse to one record per ticker-month."""
    df = pd.read_csv(path, low_memory=False)
    if "ticker" not in df.columns or "date" not in df.columns:
        raise ValueError("Gold panel must include 'ticker' and 'date' columns.")

    df = first_trading_day_per_month(df, date_col="date")

    df["month"] = pd.to_datetime(df["date"], errors="coerce").dt.to_period("M").astype(str)
    df["ticker"] = df["ticker"].astype(str).str.upper().str.strip()

    if "adj_close" in df.columns:
        df["last_price"] = pd.to_numeric(df["adj_close"], errors="coerce")
    elif "price" in df.columns:
        df["last_price"] = pd.to_numeric(df["price"], errors="coerce")
    elif "price_avg" in df.columns:
        df["last_price"] = pd.to_numeric(df["price_avg"], errors="coerce")

    return df


def detect_mode_from_folder(folder: Path) -> str:
    """Detect prediction mode using filename patterns."""
    any_workflow = any(folder.glob("*_workflow_output_*.json"))
    any_agent = any(folder.glob("*_output_*.json"))
    if any_workflow and not any_agent:
        return "workflow"
    if any_agent and not any_workflow:
        return "agent"
    if any_workflow:
        return "workflow"
    return "agent"


## 2. Load predictions

In [4]:
def load_predictions_agent(folder: Path) -> pd.DataFrame:
    """Load agent outputs into a flat indicator table."""
    rows: List[Dict[str, Any]] = []

    for p in sorted(folder.glob("*_output_*.json")):
        payload = json.loads(p.read_text(encoding="utf-8"))
        ticker = str(payload.get("ticker", "")).upper().strip()
        outputs = payload.get("outputs", [])

        if not isinstance(outputs, list):
            continue

        for item in outputs:
            date = item.get("date")
            analyst = item.get("analyst", {}) or {}
            indicators = analyst.get("indicators", {}) or {}

            if not date or not isinstance(indicators, dict):
                continue

            dt = pd.to_datetime(str(date), errors="coerce")
            if pd.isna(dt):
                continue

            month = str(dt.to_period("M"))

            for k, v in indicators.items():
                rows.append(
                    {
                        "ticker": ticker,
                        "date": dt.strftime("%Y-%m-%d"),
                        "month": month,
                        "indicator": str(k).strip(),
                        "pred_value": v,
                    }
                )

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    df["ticker"] = df["ticker"].astype(str).str.upper().str.strip()
    df["indicator"] = df["indicator"].astype(str).str.strip()
    df["pred_value"] = pd.to_numeric(df["pred_value"], errors="coerce")

    df = df.sort_values(["ticker", "indicator", "date"]).reset_index(drop=True)
    df = df.drop_duplicates(subset=["ticker", "month", "indicator"], keep="first").reset_index(drop=True)
    return df


def load_actions_workflow(folder: Path) -> pd.DataFrame:
    """Load workflow outputs into a flat action table."""
    rows: List[Dict[str, Any]] = []

    for p in sorted(folder.glob("*_workflow_output_*.json")):
        payload = json.loads(p.read_text(encoding="utf-8"))
        ticker = str(payload.get("ticker", "")).upper().strip()
        outputs = payload.get("outputs", [])

        if not isinstance(outputs, list):
            continue

        for item in outputs:
            date = item.get("date")
            manager = item.get("manager", {}) or {}
            if not date:
                continue

            dt = pd.to_datetime(str(date), errors="coerce")
            if pd.isna(dt):
                continue

            rec = manager.get("action")
            if rec is None:
                rec = manager.get("recommendation")

            if isinstance(rec, str):
                rec_u = rec.strip().upper()
            else:
                rec_u = "HOLD"

            if rec_u == "KEEP":
                rec_u = "HOLD"
            if rec_u not in {"BUY", "SELL", "HOLD"}:
                rec_u = "HOLD"

            rows.append(
                {
                    "ticker": ticker,
                    "date": dt.strftime("%Y-%m-%d"),
                    "month": str(dt.to_period("M")),
                    "action": rec_u,
                }
            )

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    df["ticker"] = df["ticker"].astype(str).str.upper().str.strip()
    df["date_dt"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.dropna(subset=["date_dt"]).sort_values(["ticker", "date_dt"]).reset_index(drop=True)
    return df.drop(columns=["date_dt"])


## 3. Stage 1. Thiago style MAE

In [5]:
def mae_list_minmax_fit_gold(y_true: pd.Series, y_pred: pd.Series) -> List[float]:
    """Per point absolute errors after MinMax scaling fitted on gold values."""
    t = pd.to_numeric(y_true, errors="coerce").to_numpy()
    p = pd.to_numeric(y_pred, errors="coerce").to_numpy()

    mask = ~np.isnan(t) & ~np.isnan(p)
    t = t[mask]
    p = p[mask]

    if t.size == 0:
        return []

    scaler = MinMaxScaler()
    scaler.fit(t.reshape(-1, 1))

    t_s = scaler.transform(t.reshape(-1, 1)).reshape(-1)
    p_s = scaler.transform(p.reshape(-1, 1)).reshape(-1)

    return [float(abs(t_s[i] - p_s[i])) for i in range(len(t_s))]


def stage1_agent_thiago_style(gold: pd.DataFrame, pred_indicators: pd.DataFrame, terms: List[str]) -> pd.DataFrame:
    """Per term mean MAE for agent mode, Thiago style."""
    rows: List[Dict[str, Any]] = []

    for term in terms:
        if term not in gold.columns:
            continue

        g = gold[["ticker", "month", term]].rename(columns={term: "gold"}).copy()
        p = (
            pred_indicators[pred_indicators["indicator"] == term][["ticker", "month", "pred_value"]]
            .rename(columns={"pred_value": "pred"})
            .copy()
        )

        merged = g.merge(p, on=["ticker", "month"], how="inner")
        if merged.empty:
            continue

        mae_list = mae_list_minmax_fit_gold(merged["gold"], merged["pred"])
        rows.append(
            {
                "term": term,
                "n_points": int(len(mae_list)),
                "mean_mae": float(np.mean(mae_list)) if len(mae_list) > 0 else float("nan"),
            }
        )

    return pd.DataFrame(rows).sort_values(["mean_mae", "term"], na_position="last").reset_index(drop=True)


## 4. Stage 2. Trading simulation

In [6]:
def simulate_simple_trades(actions: pd.DataFrame, risk_free_rate_annual: float = 0.0) -> pd.DataFrame:
    """Simulate simple monthly trading from BUY, SELL, HOLD actions."""
    if actions.empty:
        return pd.DataFrame()

    rf_monthly = float(risk_free_rate_annual) / 12.0
    results: List[Dict[str, Any]] = []

    for ticker, df in actions.groupby("ticker", sort=True):
        df = df.copy()
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
        df = df.dropna(subset=["date"]).sort_values("date").reset_index(drop=True)

        open_buys: List[float] = []
        equity = 1.0
        monthly_rets: List[float] = []

        n_buys = 0
        n_sells = 0
        n_trades_closed = 0

        for _, row in df.iterrows():
            price = row.get("price")
            action = row.get("action", "HOLD")

            if pd.isna(price) or float(price) <= 0:
                monthly_rets.append(0.0)
                continue

            if action == "BUY":
                open_buys.append(float(price))
                n_buys += 1
                monthly_rets.append(0.0)
                continue

            if action == "SELL":
                n_sells += 1
                if open_buys:
                    avg_buy = float(np.mean(open_buys))
                    r = (float(price) - avg_buy) / avg_buy
                    equity *= (1.0 + r)
                    open_buys = []
                    n_trades_closed += 1
                    monthly_rets.append(float(r))
                else:
                    monthly_rets.append(0.0)
                continue

            monthly_rets.append(0.0)

        if len(df) > 0:
            last_price = df.iloc[-1]["price"]
            if open_buys and not pd.isna(last_price) and float(last_price) > 0:
                avg_buy = float(np.mean(open_buys))
                r = (float(last_price) - avg_buy) / avg_buy
                equity *= (1.0 + r)
                n_trades_closed += 1
                monthly_rets[-1] = float(monthly_rets[-1] + r)
                open_buys = []

        cum_return = float(equity - 1.0)

        rets = np.array(monthly_rets, dtype=float)
        excess = rets - rf_monthly

        std = float(np.std(excess, ddof=1)) if len(excess) > 1 else 0.0
        mean = float(np.mean(excess)) if len(excess) > 0 else 0.0
        sharpe = float((mean / std) * math.sqrt(12.0)) if std > 0 else float("nan")

        results.append(
            {
                "ticker": ticker,
                "cumulative_return": cum_return,
                "sharpe_ratio": sharpe,
                "n_months": int(len(df)),
                "n_buys": int(n_buys),
                "n_sells": int(n_sells),
                "n_trades_closed": int(n_trades_closed),
            }
        )

    return pd.DataFrame(results).sort_values("ticker").reset_index(drop=True)


In [7]:
# check.here

## 5. Configuration

Edit these paths and settings as needed, then run the notebook top to bottom.

In [27]:
GOLD_CSV = Path("data/processed/panel/monthly_panel_prices_returns_fundamentals.csv")
# "tickers": ["TSLA", "AMZN", "NIO", "MSFT", "AAPL", "GOOG", "NFLX", "COIN"],
# For agent: results/experiments/monthly_workflow
# For workflow: results/experiments/monthly_workflow_workflow
PRED_DIR = Path("results/experiments/monthly_workflow_US2025")

# Mode: "auto", "agent", "workflow"
MODE = "auto"

# Stage 1 terms for US panel
STAGE1_TERMS = ["Revenues", "NetIncomeLoss", "Assets", "Liabilities", "last_price"]

# Stage 2
RISK_FREE_RATE_ANNUAL = 0.0


## 6. Run evaluation

In [28]:
gold = load_gold_monthly_panel(GOLD_CSV)
print("Gold rows:", len(gold))
print("Gold tickers:", gold["ticker"].nunique())
gold.tail()


Gold rows: 384
Gold tickers: 8


,ticker,date,adj_close,ret_1d,log_ret_1d,Volume,filed,Assets,Liabilities,StockholdersEquity,...,EarningsPerShareBasic,CommonStockSharesOutstanding,price_open,price_close,price_avg,price_min,price_max,price_volume,month,last_price
379,TSLA,2025-08-01,302.630005,-0.018296,-0.018465,89121400,2025-07-24,1.285670e+11,5.049500e+10,7.731400e+10,...,0.36,3.224000e+09,NaN,NaN,302.630005,NaN,NaN,89121400,2025-08,302.630005
380,TSLA,2025-09-02,329.359985,-0.013508,-0.013600,58392000,2025-07-24,1.285670e+11,5.049500e+10,7.731400e+10,...,0.36,3.224000e+09,NaN,NaN,329.359985,NaN,NaN,58392000,2025-09,329.359985
381,TSLA,2025-10-01,459.459991,0.033144,0.032607,98122300,2025-07-24,1.285670e+11,5.049500e+10,7.731400e+10,...,0.36,3.224000e+09,NaN,NaN,459.459991,NaN,NaN,98122300,2025-10,459.459991
382,TSLA,2025-11-03,468.369995,0.025867,0.025538,84595200,2025-10-23,1.337350e+11,5.301900e+10,7.997000e+10,...,0.43,3.324000e+09,NaN,NaN,468.369995,NaN,NaN,84595200,2025-11,468.369995
383,TSLA,2025-12-01,430.140015,-0.000070,-0.000070,57463600,2025-10-23,1.337350e+11,5.301900e+10,7.997000e+10,...,0.43,3.324000e+09,NaN,NaN,430.140015,NaN,NaN,57463600,2025-12,430.140015


In [29]:
mode = MODE
if mode == "auto":
    mode = detect_mode_from_folder(PRED_DIR)

print("Mode:", mode)
print("Prediction dir:", str(PRED_DIR))


Mode: workflow
Prediction dir: results/experiments/monthly_workflow_US2025


In [22]:
if mode == "agent":
    PRED_DIR = Path("results/experiments/monthly_agent_workflow")
    pred_indicators = load_predictions_agent(PRED_DIR)
    if pred_indicators.empty:
        raise RuntimeError("No agent predictions found in {0}".format(str(PRED_DIR)))

    per_term = stage1_agent_thiago_style(gold, pred_indicators, STAGE1_TERMS)
    display(per_term)
    print("Overall mean of per term means:", float(per_term["mean_mae"].mean()) if not per_term.empty else float("nan"))

elif mode == "workflow":
    print("Stage 1 skipped for workflow outputs.")
else:
    raise ValueError("Unknown mode: {0}".format(mode))


,term,n_points,mean_mae
0,Assets,12,0.070594


Overall mean of per term means: 0.07059387975854015


In [30]:
if mode == "workflow":
    PRED_DIR = Path("results/experiments/monthly_workflow_US2025")
    actions = load_actions_workflow(PRED_DIR)
    if actions.empty:
        raise RuntimeError("No workflow actions found in {0}".format(str(PRED_DIR)))

    gold_price = gold[["ticker", "month", "last_price"]].rename(columns={"last_price": "price"}).copy()
    actions = actions.merge(gold_price, on=["ticker", "month"], how="left")

    trade_df = simulate_simple_trades(actions, risk_free_rate_annual=RISK_FREE_RATE_ANNUAL)
    display(trade_df)

    print("Monthly points (gold):", len(gold))
    print("Monthly points (pred):", len(actions.drop_duplicates(subset=["ticker", "month"])))


,ticker,cumulative_return,sharpe_ratio,n_months,n_buys,n_sells,n_trades_closed
0,TSLA,0.000428,0.196197,12,3,6,2


Monthly points (gold): 384
Monthly points (pred): 12
